# Demos: Lecture 02

In [ ]:
import pennylane as qp
from pennylane import numpy as np
import matplotlib.pyplot as plt

## Demo 1: bit flip code

In [ ]:
dev = qp.device("default.mixed", wires=3)

In [ ]:
def prepare_state(wires):
    #qp.RY(np.pi/3, wires=wires)
    qp.Identity(wires=0)
    
def encode_bit_flip(wires):
    qp.CNOT(wires=[wires[0], wires[1]])
    qp.CNOT(wires=[wires[0], wires[2]])

def detect_and_recover_bit_flip(wires):
    qp.CNOT(wires=[wires[0], wires[2]])
    qp.CNOT(wires=[wires[0], wires[1]])
    qp.Toffoli(wires=wires[::-1])

@qp.qnode(dev)
def bit_flip_code(bit_to_flip=None):
    prepare_state(wires=0)
    encode_bit_flip(wires=dev.wires)
    if bit_to_flip is not None:
        qp.PauliX(wires=bit_to_flip)
    detect_and_recover_bit_flip(dev.wires)
    return qp.state()

In [ ]:
qp.draw_mpl(bit_flip_code)(bit_to_flip=1)

In [ ]:
qp.math.partial_trace(bit_flip_code(1), indices=[1, 2])

In [ ]:
p_values = np.linspace(0., 1, 50)

plt.plot(p_values, 1 - p_values, label="Uncorrected")
plt.plot(p_values, (1 - p_values) ** 3 + 3 * p_values * (1 - p_values) ** 2, label="Corrected")
plt.xlabel("Probability of bit flip (p)")
plt.ylabel("Fidelity (lower bound)")
plt.legend()

## Demo 2: logical operations

In [ ]:
def logical_x(wires):
    for wire in wires:
        qp.PauliX(wires=wire)

def logical_z(wires):
    qp.PauliZ(wires=wires[0])

def logical_hadamard(wires):
    qp.Hadamard(wires=wires[0])
    qp.CNOT(wires=[wires[0], wires[1]])
    qp.CNOT(wires=[wires[0], wires[2]])

@qp.qnode(dev)
def bit_flip_logical(logical_op, bit_to_flip=None):    
    encode_bit_flip(dev.wires)

    logical_x(dev.wires)
    logical_op(dev.wires)

    # Inserting an error
    if bit_to_flip is not None:
        qp.PauliX(wires=bit_to_flip)
    
    detect_and_recover_bit_flip(dev.wires)
    
    return qp.state()

In [ ]:
qp.math.partial_trace(bit_flip_logical(logical_hadamard), indices=[1, 2])

In [ ]:
def logical_cnot(wires_ctrl, wires_targ):
    for ctrl, targ in zip(wires_ctrl, wires_targ):
        qp.CNOT(wires=[ctrl, targ])

dev = qp.device("default.mixed", wires=6)

ctrl_wires = dev.wires[:3]
targ_wires = dev.wires[3:]

@qp.qnode(dev)
def apply_logical_cnot(bit_to_flip=None):
    encode_bit_flip(ctrl_wires)
    encode_bit_flip(targ_wires)
    
    logical_x(wires=ctrl_wires)
    logical_cnot(ctrl_wires, targ_wires)

    # Inserting an error
    if bit_to_flip is not None:
        qp.PauliX(wires=bit_to_flip)

    detect_and_recover_bit_flip(ctrl_wires)
    detect_and_recover_bit_flip(targ_wires)
    
    return qp.probs(wires=[ctrl_wires[0], targ_wires[0]])

In [ ]:
qp.draw_mpl(apply_logical_cnot)(2)

In [ ]:
apply_logical_cnot(bit_to_flip=5)

## Demo 3: phase flip code

In [ ]:
def prepare_state(wires):
    #qp.RY(np.pi/3, wires=wires)
    qp.Hadamard(wires=0)
    #qp.PauliX(wires=0)

def encode_phase_flip(wires):
    qp.CNOT(wires=[wires[0], wires[1]])
    qp.CNOT(wires=[wires[0], wires[2]])
    for wire in wires:
        qp.Hadamard(wires=wire)

def detect_and_recover_phase_flip(wires):
    for wire in wires:
        qp.Hadamard(wires=wire)
    qp.CNOT(wires=[wires[0], wires[2]])
    qp.CNOT(wires=[wires[0], wires[1]])
    qp.Toffoli(wires=wires[::-1])

dev = qp.device("default.mixed", wires=3)

@qp.qnode(dev)
def phase_flip_code(phase_to_flip=None):
    prepare_state(wires=0)
    encode_phase_flip(wires=dev.wires)
    if phase_to_flip is not None:
        qp.PauliZ(wires=phase_to_flip)
    detect_and_recover_phase_flip(dev.wires)
    return qp.state()

In [ ]:
qp.draw_mpl(phase_flip_code)(0)

In [ ]:
qp.math.partial_trace(phase_flip_code(0), indices=[1, 2])

## Demo 4: Shor 9-qubit code

In [ ]:
def prepare_state(wires):
    qp.RY(np.pi/3, wires=wires)
    #qp.Hadamard(wires=0)

def encode_shor(wires):
    encode_phase_flip([wires[0], wires[3], wires[6]])
    encode_bit_flip(wires[:3])
    encode_bit_flip(wires[3:6])
    encode_bit_flip(wires[6:])

def detect_and_recover_shor(wires):
    detect_and_recover_bit_flip(wires[:3])
    detect_and_recover_bit_flip(wires[3:6])
    detect_and_recover_bit_flip(wires[6:])
    detect_and_recover_phase_flip([wires[0], wires[3], wires[6]])


dev = qp.device("default.mixed", wires=9)

@qp.qnode(dev)
def shor_code(bit_to_flip=None, phase_to_flip=None):
    prepare_state(wires=0)
    
    encode_shor(wires=dev.wires)

    # Errors
    if bit_to_flip is not None:
        qp.PauliX(wires=bit_to_flip)
    if phase_to_flip is not None:
        qp.PauliZ(wires=phase_to_flip)
    
    detect_and_recover_shor(dev.wires)
    return qp.state()

In [ ]:
qp.draw_mpl(shor_code)()

In [ ]:
qp.math.partial_trace(shor_code(4, 5), indices=list(range(1, 9)))